# ASCA AI - Demand-Supply Matcher Agent Testing Notebook

This notebook tests **Demand-Supply Matcher Agent** (`src/agents/matcher.py`) and its **FEFO Risk Engine + ChromaDB Vector Store Tool** (`src/agents/tools/matcher_tool.py`).
It features **surplus volume waterfall pool allocation** (`remaining_surplus -= allocated`), ChromaDB vector disk I/O semaphore limiting, and multi-level FEFO sorting.

In [1]:
import sys
import asyncio
from pathlib import Path

# Add project root to sys.path
project_root = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src.agents.guardrail import MarketInsight, RiskLevel
from src.agents.matcher import matcher_agent
from src.agents.tools.matcher_tool import chroma_b2b_store, FEFORiskEngine

print("Demand-Supply Matcher Agent & ChromaDB tools loaded successfully!")

2026-07-30 13:03:16 | INFO     | src.agents.tools.matcher_tool:_ensure_seed_buyers:97 - Seeding initial Sri Lankan B2B Processing Plants into ChromaDB...
2026-07-30 13:03:18 | INFO     | src.agents.tools.matcher_tool:_ensure_seed_buyers:155 - Successfully seeded 4 B2B buyers into ChromaDB.
Demand-Supply Matcher Agent & ChromaDB tools loaded successfully!


## Step 1: Query ChromaDB Vector Store for Processing Plants

In [2]:
print("Querying ChromaDB Vector Store for Tomato B2B Buyers...")
buyers = chroma_b2b_store.search_buyers_for_crop("tomato", top_k=3)

print(f"Found {len(buyers)} Matched B2B Processing Plants:")
for b in buyers:
    print(f" - Code: {b['buyer_code']:<25} | Name: {b['company_name']:<30} | Location: {b['location']:<25} | Capacity: {b['daily_capacity_tons']} Tons/day")

Querying ChromaDB Vector Store for Tomato B2B Buyers...
Found 3 Matched B2B Processing Plants:
 - Code: BUYER_CANNING_MATALE      | Name: Central Province Canning Mills | Location: Matale Processing Zone    | Capacity: 25.0 Tons/day
 - Code: BUYER_SAUCE_DAMBULLA      | Name: Lanka Canning & Sauce Ltd      | Location: Dambulla Industrial Zone  | Capacity: 35.0 Tons/day
 - Code: BUYER_DEHYDRATION_KURUNEGALA | Name: Wayamba Food Dehydration Corp  | Location: Kurunegala Food Hub       | Capacity: 30.0 Tons/day


## Step 2: Test Surplus Volume Waterfall Allocation & FEFO Matching

In [3]:
# Simulate a high-risk 50 Metric Ton Tomato surplus insight
tomato_surplus_insight = MarketInsight(
    center_id="DAMBULLA",
    crop_name="tomato",
    current_wholesale_price_lkr=240.0,
    predicted_wholesale_price_lkr=120.0,
    supply_volume_tons=50.0, # 50 Tons total surplus
    surplus_anomaly_detected=True,
    risk_level=RiskLevel.HIGH
)

print("🚀 Running DemandSupplyMatcherAgent with Volume Waterfall Allocation...\n")
matches = await matcher_agent.match_surplus_crops_async([tomato_surplus_insight])

print(f"\n--- WATERFALL B2B MATCH RECOMMENDATIONS ({len(matches)} Allocations) ---")
total_allocated = 0.0
for m in matches:
    total_allocated += m.matched_volume_tons
    print(f"Buyer: {m.company_name:<30} | Allocated Volume: {m.matched_volume_tons:<4.1f} T | FEFO Score: {m.fefo_risk_score:<4.2f}")
    print(f"Action: {m.recommended_action}\n")

print(f"Total Surplus Volume Allocated: {total_allocated} / 50.0 Metric Tons")

🚀 Running DemandSupplyMatcherAgent with Volume Waterfall Allocation...

2026-07-30 13:03:25 | INFO     | src.agents.matcher:match_surplus_crops_async:94 - DemandSupplyMatcherAgent matching 1 surplus anomalies concurrently...
2026-07-30 13:03:25 | INFO     | src.agents.matcher:match_single_insight_async:27 - DemandSupplyMatcherAgent finding B2B buyers for tomato surplus at DAMBULLA (50.0 Tons)...
2026-07-30 13:03:26 | INFO     | src.agents.matcher:match_surplus_crops_async:100 - DemandSupplyMatcherAgent generated 2 total B2B Match Recommendations.

--- WATERFALL B2B MATCH RECOMMENDATIONS (2 Allocations) ---
Buyer: Lanka Canning & Sauce Ltd      | Allocated Volume: 25.0 T | FEFO Score: 0.51
Action: Route 25.0 tons of excess tomato from DAMBULLA to Lanka Canning & Sauce Ltd (Dambulla Industrial Zone). FEFO Risk Score: 0.51.

Buyer: Central Province Canning Mills | Allocated Volume: 25.0 T | FEFO Score: 0.67
Action: Route 25.0 tons of excess tomato from DAMBULLA to Central Province Canning